In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Edge AI Challenge 2026 – Lần 2 (Warm-up)

**Tuân thủ thể lệ:** 1 mô hình detection (1 lần suy luận, không ensemble/TTA, `imgsz=160`, ≤ 200.000 tham số, export TFLite INT8 ≤ 1,5 MB) + 1 mô hình Intent **TextCNN** (INT8, ≤ 1,5 MB). Một file `submission.csv` duy nhất, tối đa 100 bbox/ảnh.

**Khai báo dữ liệu / mô hình ngoài:** không dùng dữ liệu ngoài, không dùng trọng số pretrained. Tập test chỉ dùng để dự đoán.

# 1. Khai báo dataset

In [ ]:
import pandas as pd
from pathlib import Path

root = Path("/kaggle/input/competitions/edge-ai-challenge-2026-lan-2")

train = pd.read_csv(root / "intent_train.csv")
test  = pd.read_csv(root / "intent_test.csv")
sub   = pd.read_csv(root / "sample_submission.csv")

train_imgs = sorted((root / "train/images").glob("*.png"))
train_lbls = sorted((root / "train/labels").glob("*.txt"))
test_imgs  = sorted((root / "test/images").glob("*.png"))

print(len(train_imgs), len(train_lbls), len(test_imgs))

img_ids = {p.stem for p in train_imgs}
lbl_ids = {p.stem for p in train_lbls}
print("Ảnh thiếu label:", len(img_ids - lbl_ids))
print("Label thiếu ảnh:", len(lbl_ids - img_ids))

In [ ]:
print(train.shape, test.shape, sub.shape)
print(train.head())
print(test.head())
print(sub.head())

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open(train_imgs[0])
print(img.size, img.mode)
plt.imshow(img); plt.axis("off"); plt.show()

# 2. Đọc danh sách class

In [ ]:
with open(root / "classes.txt", encoding="utf-8") as f:
    classes = [line.strip() for line in f if line.strip()]

print("Số lớp:", len(classes))
print(classes)

Gán số từ 0 đến nc-1 theo đúng thứ tự trong `classes.txt` (thể lệ: 7 lớp, class_id 0–6).

In [ ]:
id2name = {i: name for i, name in enumerate(classes)}
name2id = {name: i for i, name in enumerate(classes)}

print(id2name)

In [ ]:
import numpy as np

all_ids = []
for p in train_lbls:
    arr = np.loadtxt(p, ndmin=2)
    if arr.size:                      # bỏ qua file label rỗng
        all_ids.extend(arr[:, 0].astype(int))

all_ids = np.array(all_ids)
print("id nhỏ nhất / lớn nhất:", all_ids.min(), all_ids.max())

# đếm số box mỗi lớp
from collections import Counter
cnt = Counter(all_ids)
for i in sorted(cnt):
    print(f"{i:3d} {id2name[i]:20s} {cnt[i]}")

# 3. Check dataset

check 3 file csv

In [ ]:
print(train.shape, test.shape, sub.shape)
display(train.head())
display(test.head())
display(sub.head())

print("Giá trị thiếu (train):\n", train.isnull().sum())
print("Dòng trùng (train):", train.duplicated().sum())
print("Cột train:", train.columns.tolist())
print("Cột test :", test.columns.tolist())
print("Cột sub  :", sub.columns.tolist())

check file labels

In [ ]:
import numpy as np

bad = []
empty = []
n_boxes = {}

for p in train_lbls:
    lines = [l.split() for l in open(p) if l.strip()]
    if not lines:
        empty.append(p.stem)
        continue
    n_boxes[p.stem] = len(lines)
    for parts in lines:
        if len(parts) != 5:
            bad.append((p.stem, "sai số cột", parts)); continue
        c, x, y, w, h = int(parts[0]), *map(float, parts[1:])
        if not (0 <= c < len(classes)):
            bad.append((p.stem, "class_id ngoài khoảng", c))
        if not all(0 <= v <= 1 for v in (x, y, w, h)):
            bad.append((p.stem, "toạ độ ngoài [0,1]", (x, y, w, h)))
        if w <= 0 or h <= 0:
            bad.append((p.stem, "box rỗng", (w, h)))

print("File label rỗng:", len(empty))
print("Dòng lỗi:", len(bad))
for b in bad[:10]:
    print(b)
print("Số box/ảnh:", pd.Series(n_boxes).describe())

check img 

In [ ]:
from PIL import Image
from collections import Counter

sizes, modes, corrupt = Counter(), Counter(), []

for p in train_imgs + test_imgs:
    try:
        with Image.open(p) as im:
            im.verify()
        with Image.open(p) as im:
            sizes[im.size] += 1
            modes[im.mode] += 1
    except Exception as e:
        corrupt.append((p.name, str(e)))

print("Ảnh hỏng:", len(corrupt), corrupt[:5])
print("Kích thước:", sizes.most_common(5))
print("Mode:", modes)

check img + label

In [ ]:
test_ids = [p.stem for p in test_imgs]
print("Ví dụ id test:", test_ids[:9])
print("Số ảnh test:", len(test_ids), "| số dòng sub:", len(sub))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def show(idx, ax):
    p = train_imgs[idx]
    img = Image.open(p)
    W, H = img.size
    ax.imshow(img); ax.axis("off"); ax.set_title(p.stem)
    lbl = root / "train/labels" / f"{p.stem}.txt"
    for line in open(lbl):
        c, x, y, w, h = map(float, line.split())
        ax.add_patch(patches.Rectangle(((x-w/2)*W, (y-h/2)*H), w*W, h*H,
                                       fill=False, edgecolor="red", lw=2))
        ax.text((x-w/2)*W, (y-h/2)*H, classes[int(c)], color="white",
                bbox=dict(facecolor="red", alpha=0.6), fontsize=8)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, i in zip(axes.ravel(), np.random.choice(len(train_imgs), 6, replace=False)):
    show(i, ax)
plt.tight_layout(); plt.show()

# 4. Check YOLO

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter

with open(root / "classes.txt", encoding="utf-8") as f:
    classes = [l.strip() for l in f if l.strip()]
nc = len(classes)

rows, errors, empty = [], [], []

for p in train_lbls:
    lines = [l.strip() for l in open(p) if l.strip()]
    if not lines:
        empty.append(p.stem)
        continue
    for ln, line in enumerate(lines, 1):
        parts = line.split()
        if len(parts) != 5:
            errors.append((p.stem, ln, "sai số cột", len(parts)))
            continue
        try:
            c = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:])
        except ValueError:
            errors.append((p.stem, ln, "không phải số", line))
            continue
        if not (0 <= c < nc):
            errors.append((p.stem, ln, "class_id ngoài khoảng", c))
        if not (0 <= x <= 1 and 0 <= y <= 1):
            errors.append((p.stem, ln, "tâm ngoài [0,1]", (x, y)))
        if not (0 < w <= 1 and 0 < h <= 1):
            errors.append((p.stem, ln, "w/h không hợp lệ", (w, h)))
        if x - w/2 < -1e-3 or x + w/2 > 1 + 1e-3 or y - h/2 < -1e-3 or y + h/2 > 1 + 1e-3:
            errors.append((p.stem, ln, "box tràn ra ngoài ảnh", (x, y, w, h)))
        rows.append([p.stem, c, x, y, w, h])

df = pd.DataFrame(rows, columns=["id", "cls", "x", "y", "w", "h"])
df["name"] = df["cls"].map(lambda i: classes[i] if 0 <= i < nc else "?")
df["area"] = df["w"] * df["h"]

print("Số lớp:", nc)
print("Tổng số box:", len(df), "| số ảnh có box:", df["id"].nunique())
print("File label rỗng:", len(empty), empty[:5])
print("Số dòng lỗi:", len(errors))
for e in errors[:10]:
    print("  ", e)

In [ ]:
dup = df.duplicated(subset=["id", "cls", "x", "y", "w", "h"]).sum()
print("Box trùng hệt nhau:", dup)

In [ ]:
cnt = df["name"].value_counts()
print(cnt)

import matplotlib.pyplot as plt
cnt.plot(kind="bar", figsize=(10, 4), title="Số box mỗi lớp")
plt.show()

print("Lớp không có box nào:", [c for c in classes if c not in set(df["name"])])

In [ ]:
print(df.groupby("id").size().describe())      # số object mỗi ảnh
print(df[["w", "h", "area"]].describe())        # kích thước box (chuẩn hoá)

print("Box rất nhỏ (area < 0.001):", (df["area"] < 0.001).sum())

In [ ]:
img_ids = {p.stem for p in train_imgs}
print("Ảnh không có dòng box nào:", len(img_ids - set(df["id"])))

In [ ]:
big = df[df["w"] > 0.5]
print(big)
print(big["name"].value_counts())

In [ ]:
# Kiểm tra box quá lớn / tràn ngoài ảnh theo từng lớp (không phụ thuộc tên lớp cụ thể)
for name, g in df.groupby("name"):
    left  = ((g["x"] - g["w"]/2) < -1e-3).sum()
    right = ((g["x"] + g["w"]/2) > 1 + 1e-3).sum()
    print(f"{name:16s} n={len(g):4d} | w>=0.99: {(g['w'] >= 0.99).sum():3d} | tràn trái/phải: {left}/{right}")

In [ ]:
print(df["name"].value_counts())
print("Số dòng lỗi:", len(errors), "| label rỗng:", len(empty))
print(Image.open(train_imgs[0]).size)

# 5. Trực quan hóa 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

cmap = plt.get_cmap("tab10")
colors = {i: cmap(i) for i in range(len(classes))}

def show(idx, ax):
    p = train_imgs[idx]
    img = Image.open(p).convert("RGB")
    W, H = img.size
    ax.imshow(img); ax.axis("off"); ax.set_title(p.stem)
    for line in open(root / "train/labels" / f"{p.stem}.txt"):
        c, x, y, w, h = map(float, line.split())
        c = int(c)
        x1, y1 = (x - w/2) * W, (y - h/2) * H
        ax.add_patch(patches.Rectangle((x1, y1), w*W, h*H, fill=False,
                                       edgecolor=colors[c], lw=2))
        ax.text(x1, y1, classes[c], color="white", fontsize=8,
                bbox=dict(facecolor=colors[c], alpha=0.7, pad=1))

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, i in zip(axes.ravel(), np.random.choice(len(train_imgs), 6, replace=False)):
    show(i, ax)
plt.tight_layout(); plt.show()

In [ ]:
first_per_class = df.groupby("cls")["id"].first()
ids = {p.stem: i for i, p in enumerate(train_imgs)}

ncols = 5
nrows = -(-len(first_per_class) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(24, 4 * nrows))
axes = np.atleast_1d(axes).ravel()
for ax in axes:
    ax.axis("off")
for ax, (c, img_id) in zip(axes, first_per_class.items()):
    show(ids[img_id], ax)
    ax.set_title(f"{classes[c]} ({img_id})")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

df["name"].value_counts().plot(kind="bar", ax=axes[0], title="Số box mỗi lớp")
df.groupby("id").size().plot(kind="hist", bins=12, ax=axes[1], title="Số box mỗi ảnh")
axes[2].scatter(df["w"], df["h"], c=df["cls"].map(lambda i: colors[i]), s=12)
axes[2].set_xlabel("w"); axes[2].set_ylabel("h"); axes[2].set_title("Kích thước box (chuẩn hoá)")

plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.hist2d(df["x"], 1 - df["y"], bins=30, cmap="viridis")
plt.colorbar(label="số box"); plt.title("Mật độ tâm box"); plt.show()

# 6. Chia train/val ttheo class 

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

cls_count = df["cls"].value_counts()
img_cls = (df.groupby("id")["cls"]
             .apply(lambda s: min(set(s), key=lambda c: cls_count[c])))

ids = img_cls.index.to_numpy()
strat = img_cls.to_numpy()

# nhóm nào < 2 ảnh thì không stratify được -> đưa thẳng vào train
vc = pd.Series(strat).value_counts()
rare = set(vc[vc < 2].index)
mask = np.array([s in rare for s in strat])

ids_rare = ids[mask]
ids_ok, strat_ok = ids[~mask], strat[~mask]

train_ids, val_ids = train_test_split(
    ids_ok, test_size=0.2, random_state=42, stratify=strat_ok
)
train_ids = list(train_ids) + list(ids_rare)

no_box = sorted({p.stem for p in train_imgs} - set(ids))
train_ids = train_ids + no_box

print("Train:", len(train_ids), "| Val:", len(val_ids))
print("Ảnh của nhóm hiếm đưa vào train:", len(ids_rare))

In [ ]:
# Không chia lại train_ids / val_ids ở cell này.
# Cell 35 đã chia stratified và đưa ảnh không có box vào train.
print("Giữ nguyên phân chia từ Cell 35")
print("Train:", len(train_ids), "| Val:", len(val_ids))
print("Ảnh giao nhau:", len(set(train_ids) & set(val_ids)))


In [ ]:
d_tr = df[df["id"].isin(train_ids)]["name"].value_counts()
d_va = df[df["id"].isin(val_ids)]["name"].value_counts()

check = pd.concat([d_tr, d_va], axis=1, keys=["train", "val"]).fillna(0).astype(int)
check["val_%"] = (check["val"] / (check["train"] + check["val"]) * 100).round(1)
print(check)

In [ ]:
print(len(train_ids), len(val_ids))
print(len(set(train_ids) & set(val_ids)))   # phải bằng 0: không ảnh nào nằm ở cả hai tập

# 7. Cấu trúc YOLO 

In [ ]:
import os, shutil
from pathlib import Path

out = Path("/kaggle/working/dataset")
if out.exists():
    shutil.rmtree(out)

for split in ["train", "val"]:
    (out / "images" / split).mkdir(parents=True)
    (out / "labels" / split).mkdir(parents=True)

def put(ids, split):
    for i in ids:
        os.symlink(root / "train/images" / f"{i}.png", out / "images" / split / f"{i}.png")
        lbl = root / "train/labels" / f"{i}.txt"
        if lbl.exists():
            shutil.copy(lbl, out / "labels" / split / f"{i}.txt")

put(train_ids, "train")
put(val_ids, "val")

print("train:", len(list((out / "images/train").glob("*.png"))),
      "ảnh,", len(list((out / "labels/train").glob("*.txt"))), "label")
print("val  :", len(list((out / "images/val").glob("*.png"))),
      "ảnh,", len(list((out / "labels/val").glob("*.txt"))), "label")

In [ ]:
import yaml

data = {
    "path": str(out),
    "train": "images/train",
    "val": "images/val",
    "nc": len(classes),
    "names": {i: n for i, n in enumerate(classes)},
}
with open(out / "data.yaml", "w", encoding="utf-8") as f:
    yaml.dump(data, f, allow_unicode=True, sort_keys=False)

print(open(out / "data.yaml").read())

In [ ]:
for s in ["train", "val"]:
    ni = len(list((out / f"images/{s}").glob("*.png")))
    nl = len(list((out / f"labels/{s}").glob("*.txt")))
    print(s, ni, nl)

# 8. Huấn luyện YOLO 


In [ ]:
!pip install -q ultralytics

from pathlib import Path
import shutil, yaml, torch
import ultralytics
from ultralytics import YOLO

# ---- Ràng buộc thể lệ (mục V) ----
IMGSZ   = 160        # mỗi chiều <= 320x240; khuyến nghị 96/128/160
MAX_DET = 100        # tối đa 100 bbox dự đoán mỗi ảnh (mục IV.2)
MAX_PARAMS = 200_000
MAX_BYTES  = int(1.5 * 1024 * 1024)
print("ultralytics:", ultralytics.__version__, "| torch:", torch.__version__)   # ghi lại để tái tạo

DATA_YAML = out / "data.yaml"
TINY_YAML = Path("/kaggle/working/tiny_yolov8_edge.yaml")

# Mạng YOLO rất nhỏ, phát hiện tại stride 8 (một head P3).
# Ít tầng và ít kênh để phù hợp giới hạn tham số của cuộc thi.
tiny_cfg = {
    "nc": len(classes),
    "depth_multiple": 1.0,
    "width_multiple": 1.0,
    "backbone": [
        [-1, 1, "Conv", [16, 3, 2]],
        [-1, 1, "Conv", [24, 3, 2]],
        [-1, 1, "C2f", [24, True]],
        [-1, 1, "Conv", [32, 3, 2]],
        [-1, 1, "C2f", [32, True]],
    ],
    "head": [
        [[4], 1, "Detect", [len(classes)]]
    ]
}
with open(TINY_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(tiny_cfg, f, sort_keys=False, allow_unicode=True)

# Khởi tạo từ YAML, KHÔNG nạp trọng số pretrained (không dùng dữ liệu/model ngoài).
model = YOLO(str(TINY_YAML))
num_params = sum(p.numel() for p in model.model.parameters())
print(f"Tiny YOLO parameters: {num_params:,} / {MAX_PARAMS:,}")
if num_params > MAX_PARAMS:
    raise ValueError("Mô hình vượt giới hạn 200.000 parameters; cần thu nhỏ YAML trước khi train.")

results = model.train(
    data=str(DATA_YAML),
    imgsz=IMGSZ,
    epochs=150,
    batch=16,
    patience=30,
    seed=42,
    degrees=5.0,
    translate=0.05,
    scale=0.15,
    fliplr=0.5,
    flipud=0.0,
    mosaic=0.5,
    close_mosaic=10,
    workers=2,
    device=0 if torch.cuda.is_available() else "cpu",
    project="/kaggle/working/runs",
    name="tiny_edgeai",
    exist_ok=True,
    plots=True,
)

# Lấy đúng thư mục mà Ultralytics vừa tạo, không hard-code exp1/exp1-3.
RUN_DIR = Path(model.trainer.save_dir)
BEST_MODEL = RUN_DIR / "weights" / "best.pt"
LAST_MODEL = RUN_DIR / "weights" / "last.pt"
if not BEST_MODEL.exists():
    if LAST_MODEL.exists():
        BEST_MODEL = LAST_MODEL
    else:
        raise FileNotFoundError(f"Không tìm thấy best.pt hoặc last.pt trong {RUN_DIR}")

print("Run directory:", RUN_DIR)
print("Best model:", BEST_MODEL)

# Nạp lại checkpoint tốt nhất để các cell sau dùng cùng một model.
best_model = YOLO(str(BEST_MODEL))
print("Parameters trong checkpoint:", sum(p.numel() for p in best_model.model.parameters()))

# ---- Export TFLite INT8 (bước phụ: lỗi không làm hỏng pipeline, nhưng PHẢI kiểm tra kết quả) ----
DET_TFLITE = None
try:
    export_path = Path(str(best_model.export(format="tflite", int8=True, imgsz=IMGSZ, data=str(DATA_YAML))))
    search_dir = export_path if export_path.is_dir() else export_path.parent
    cands = sorted(search_dir.rglob("*int8*.tflite")) or sorted(search_dir.rglob("*.tflite"))
    if not cands:
        raise FileNotFoundError(f"Không thấy file .tflite trong {search_dir}")
    DET_TFLITE = Path("/kaggle/working/detect_int8.tflite")
    shutil.copy(cands[0], DET_TFLITE)

    size = DET_TFLITE.stat().st_size
    print(f"TFLite detection: {DET_TFLITE.name} | {size/1024:.1f} KB |",
          "OK (<= 1.5MB)" if size <= MAX_BYTES else "VƯỢT 1.5MB -> phải thu nhỏ mô hình")

    import tensorflow as tf
    it = tf.lite.Interpreter(model_path=str(DET_TFLITE)); it.allocate_tensors()
    i0 = it.get_input_details()[0]
    print("Input:", i0["dtype"], i0["shape"], "| Output:", [(o["dtype"], o["shape"]) for o in it.get_output_details()])
    try:
        print("Toán tử:", sorted({d["op_name"] for d in it._get_ops_details()}))
    except Exception:
        pass
except Exception as e:
    print("TFLite export chưa thành công; checkpoint best.pt vẫn được giữ. Cần xử lý trước chung kết.")
    print(type(e).__name__, str(e)[:500])

### Ghi chú – TextCNN và TFLite

TextCNN được huấn luyện bằng **PyTorch** (mục 13–14). Bản TFLite INT8 của TextCNN được tạo ở **mục 15b** (PyTorch → ONNX → TFLite INT8, có phương án dự phòng qua Keras). Không dùng đoạn Keras cũ vì `tf` / `model_txt_tf` không thuộc pipeline huấn luyện.

In [ ]:
from ultralytics import YOLO
import pandas as pd

# Dùng đúng checkpoint đã tìm thấy ở Cell 44.
assert BEST_MODEL.exists(), f"Không tìm thấy model: {BEST_MODEL}"
val_model = YOLO(str(BEST_MODEL))

m = val_model.val(
    data=str(DATA_YAML),
    imgsz=IMGSZ,
    conf=0.001,
    iou=0.6,
    device=0 if torch.cuda.is_available() else "cpu",
)

print("mAP50   :", round(m.box.map50, 3))
print("mAP50-95:", round(m.box.map, 3))
print("Precision:", round(m.box.mp, 3))
print("Recall   :", round(m.box.mr, 3))

res = pd.DataFrame({
    "class": [classes[int(i)] for i in m.box.ap_class_index],
    "AP50": m.box.ap50,
    "AP50-95": m.box.ap,
}).sort_values("AP50-95")
display(res.round(3))

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

for fname in ["results.png", "confusion_matrix_normalized.png", "BoxPR_curve.png"]:
    path = RUN_DIR / fname
    if path.exists():
        with Image.open(path) as im:
            plt.figure(figsize=(12, 6))
            plt.imshow(im.copy())
            plt.title(fname)
            plt.axis("off")
            plt.show()
    else:
        print("Không tìm thấy:", path)


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# Đánh giá trực quan trên đúng tập validation, cùng imgsz với lúc train.
val_pred_results = val_model.predict(
    source=str(out / "images" / "val"),
    imgsz=IMGSZ,
    conf=0.15,
    max_det=MAX_DET,
    device=0 if torch.cuda.is_available() else "cpu",
    save=True,
    project="/kaggle/working/runs",
    name="tiny_pred_val",
    exist_ok=True,
    verbose=False,
)

import glob
for p in sorted(glob.glob("/kaggle/working/runs/tiny_pred_val/*.jpg") +
                glob.glob("/kaggle/working/runs/tiny_pred_val/*.png"))[:6]:
    with Image.open(p) as im:
        plt.figure(figsize=(10, 6))
        plt.imshow(im.copy())
        plt.axis("off")
        plt.show()

# 9. Dự đoán tập test bằng YOLO

In [ ]:
from pathlib import Path
import pandas as pd

# Load best checkpoint và dự đoán toàn bộ ảnh test (1 model, 1 lần suy luận, không TTA/ensemble).
test_model = YOLO(str(BEST_MODEL))
test_results = test_model.predict(
    source=str(root / "test" / "images"),
    imgsz=IMGSZ,
    conf=0.001,
    iou=0.6,
    max_det=MAX_DET,
    device=0 if torch.cuda.is_available() else "cpu",
    stream=True,
    verbose=False,
)

# Đặt tên riêng det_pred_df để không bị biến pred của TextCNN ghi đè ở các cell sau.
rows = []
for r in test_results:
    img_id = Path(r.path).stem
    orig_h, orig_w = r.orig_shape
    for b in r.boxes:
        cls_id = int(b.cls.item())
        score = float(b.conf.item())
        x1, y1, x2, y2 = map(float, b.xyxy[0].cpu().tolist())
        # Giới hạn toạ độ trong ảnh gốc.
        x1 = max(0.0, min(x1, float(orig_w)))
        x2 = max(0.0, min(x2, float(orig_w)))
        y1 = max(0.0, min(y1, float(orig_h)))
        y2 = max(0.0, min(y2, float(orig_h)))
        if x2 <= x1 or y2 <= y1:
            continue
        x, y, w, h = map(float, b.xywhn[0].cpu().tolist())
        rows.append([img_id, cls_id, classes[cls_id], score,
                     x, y, w, h, x1, y1, x2, y2])

columns = ["id", "cls", "name", "conf", "x", "y", "w", "h", "x1", "y1", "x2", "y2"]
det_pred_df = pd.DataFrame(rows, columns=columns)
print("Số box dự đoán:", len(det_pred_df))
print("Ảnh test có box:", det_pred_df["id"].nunique() if len(det_pred_df) else 0,
      "/", len(test_imgs))
display(det_pred_df.head())

In [ ]:
# Kiểm tra DataFrame dự đoán YOLO.
print(type(det_pred_df), det_pred_df.shape)
print("Ảnh có box:", det_pred_df["id"].nunique() if len(det_pred_df) else 0, "/", len(test_imgs))


In [ ]:
# Ngưỡng lọc để tạo submission. mAP50 thường tốt hơn khi giữ cả box điểm thấp,
# nên dùng ngưỡng rất thấp; số box mỗi ảnh được chặn ở MAX_DET = 100 (quy định thể lệ).
CONF_THRES = 0.001
det_pred = det_pred_df[det_pred_df["conf"] >= CONF_THRES].copy()
det_pred = (det_pred.sort_values(["id", "conf"], ascending=[True, False])
                    .groupby("id", group_keys=False).head(MAX_DET))
print("Confidence threshold:", CONF_THRES, "| MAX_DET:", MAX_DET)
print("Số box sau lọc:", len(det_pred))
print("Ảnh còn box:", det_pred["id"].nunique() if len(det_pred) else 0, "/", len(test_imgs))
if len(det_pred):
    print("Số box theo lớp:")
    display(det_pred["name"].value_counts().rename_axis("class").to_frame("boxes"))
    print("Thống kê box mỗi ảnh (phải <= 100):")
    display(det_pred.groupby("id").size().describe())

In [ ]:
print(det_pred_df["name"].value_counts() if len(det_pred_df) else "Chưa có box")
if len(det_pred_df):
    print(det_pred_df.groupby("id").size().describe())
print("Ảnh test không có box:", len({p.stem for p in test_imgs} - set(det_pred_df["id"])))


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import glob

# Vẽ kết quả dự đoán trên tập test.
test_model.predict(
    source=str(root / "test" / "images"),
    imgsz=IMGSZ,
    conf=0.15,
    max_det=MAX_DET,
    device=0 if torch.cuda.is_available() else "cpu",
    save=True,
    project="/kaggle/working/runs",
    name="tiny_pred_test",
    exist_ok=True,
    verbose=False,
)

files = sorted(glob.glob("/kaggle/working/runs/tiny_pred_test/*.jpg") +
               glob.glob("/kaggle/working/runs/tiny_pred_test/*.png"))
for p in files[:4]:
    with Image.open(p) as im:
        plt.figure(figsize=(10, 6))
        plt.imshow(im.copy())
        plt.axis("off")
        plt.show()

In [ ]:
print(sub.head(10))
print(sub.columns.tolist(), sub.shape)
print(test.head())

# 10. Chuẩn bị dữ liệu intent 

In [ ]:
print(train.shape, test.shape, sub.shape)
print("Cột train:", train.columns.tolist())
print("Cột test :", test.columns.tolist())
print("Cột sub  :", sub.columns.tolist())

display(train.head(10))
display(test.head(5))
display(sub.head(5))

kiểm tra chất lượng dữ liệu 

In [ ]:
print(train.isnull().sum())
print("Dòng trùng:", train.duplicated().sum())
print(train.dtypes)

phân bố nhãn 

In [ ]:
col = "intent"
print(train[col].value_counts())
print("Số nhãn khác nhau:", train[col].nunique())

In [ ]:
text_col = "question"   
print(train[text_col].str.len().describe())
print(train.sample(5, random_state=0)[[text_col, col]])

In [ ]:
print(sub["task_type"].value_counts())
print(sub.head(5))
print(sub.tail(5))

# 11. Chia train/validation 

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

col, text_col = "intent", "question"

vc = train[col].value_counts()
print(vc)
print("Số intent:", len(vc), "| ít nhất:", vc.min())

use_strat = vc.min() >= 2 and len(vc) <= int(len(train) * 0.2)   # val phải đủ chỗ cho mỗi intent
tr_df, va_df = train_test_split(
    train, test_size=0.2, random_state=42,
    stratify=train[col] if use_strat else None,
)
tr_df, va_df = tr_df.reset_index(drop=True), va_df.reset_index(drop=True)

print("Stratify:", use_strat)
print("Train:", len(tr_df), "| Val:", len(va_df))

check = pd.concat([tr_df[col].value_counts(), va_df[col].value_counts()],
                  axis=1, keys=["train", "val"]).fillna(0).astype(int)
print(check)
print("Intent không có ở val:", list(check[check["val"] == 0].index))

In [ ]:
pd.set_option("display.max_colwidth", 120)
print(train.groupby("intent").head(3).sort_values("intent")[["question_id", "question", "intent"]])
print(test.head(10))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, accuracy_score

clf = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), analyzer="char_wb", lowercase=True),
    LogisticRegression(max_iter=1000, class_weight="balanced"),
)
clf.fit(tr_df["question"], tr_df["intent"])

pred_va = clf.predict(va_df["question"])
print("Accuracy:", accuracy_score(va_df["intent"], pred_va))
print(classification_report(va_df["intent"], pred_va, zero_division=0))

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)   # 3 vì OUT_OF_SCOPE chỉ có 3 câu
print(cross_val_score(clf, train["question"], train["intent"], cv=cv).mean())

# 12. Tạo vocabulary 

In [ ]:
import re
from collections import Counter

def tokenize(s):
    s = s.lower().strip()
    return re.findall(r"\w+", s, flags=re.UNICODE)    # giữ chữ có dấu tiếng Việt

counter = Counter()
for q in tr_df["question"]:
    counter.update(tokenize(q))

print("Số từ khác nhau:", len(counter))
print(counter.most_common(20))

In [ ]:
PAD, UNK = "<pad>", "<unk>"

min_freq = 1            # dữ liệu rất nhỏ nên giữ mọi từ; tăng lên 2 nếu có nhiều từ hiếm
itos = [PAD, UNK] + [w for w, c in counter.most_common() if c >= min_freq]
stoi = {w: i for i, w in enumerate(itos)}

print("Kích thước vocab:", len(itos))

In [ ]:
import numpy as np

def encode(s, max_len):
    ids = [stoi.get(w, stoi[UNK]) for w in tokenize(s)][:max_len]
    return ids + [stoi[PAD]] * (max_len - len(ids))

lens = tr_df["question"].map(lambda s: len(tokenize(s)))
print(lens.describe())
max_len = int(lens.max())          # hoặc percentile 95 nếu có câu rất dài

X_tr = np.array([encode(q, max_len) for q in tr_df["question"]])
X_va = np.array([encode(q, max_len) for q in va_df["question"]])

# mã hoá nhãn
intents = sorted(train["intent"].unique())
lab2id = {l: i for i, l in enumerate(intents)}
y_tr = tr_df["intent"].map(lab2id).to_numpy()
y_va = va_df["intent"].map(lab2id).to_numpy()
print(lab2id, X_tr.shape, X_va.shape)

In [ ]:
oov = sum(w not in stoi for q in va_df["question"] for w in tokenize(q))
tot = sum(len(tokenize(q)) for q in va_df["question"])
print(f"Từ ngoài vocab ở val: {oov}/{tot} = {oov/tot:.1%}")

# 13. Định nghĩa TextCNN 

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

KERNEL_SIZES = (2, 3, 4)
assert max_len >= max(KERNEL_SIZES), f"max_len={max_len} phải >= kernel lớn nhất {max(KERNEL_SIZES)}"

class TextCNN(nn.Module):
    def __init__(self, vocab_size, num_classes, pad_idx,
                 emb_dim=64, num_filters=32, kernel_sizes=KERNEL_SIZES, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.convs = nn.ModuleList([nn.Conv1d(emb_dim, num_filters, k) for k in kernel_sizes])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), num_classes)

    def forward(self, x):                        # x: (B, L)
        e = self.embedding(x).transpose(1, 2)    # (B, emb_dim, L)
        feats = [F.relu(conv(e)).max(dim=2).values for conv in self.convs]
        return self.fc(self.dropout(torch.cat(feats, dim=1)))   # (B, num_classes) - logits

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_txt = TextCNN(
    vocab_size=len(itos),
    num_classes=len(lab2id),
    pad_idx=stoi[PAD],
).to(device)

print(model_txt)
print("Số tham số:", sum(p.numel() for p in model_txt.parameters()))

xb = torch.tensor(X_tr[:4], dtype=torch.long).to(device)
print(model_txt(xb).shape)      # mong đợi: torch.Size([4, 5])

# 14. Huấn luyện TextCNN 

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

Xtr_t = torch.tensor(X_tr, dtype=torch.long)
ytr_t = torch.tensor(y_tr, dtype=torch.long)
Xva_t = torch.tensor(X_va, dtype=torch.long)
yva_t = torch.tensor(y_va, dtype=torch.long)

train_dl = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=8, shuffle=True)
val_dl   = DataLoader(TensorDataset(Xva_t, yva_t), batch_size=32)

# trọng số lớp: lớp càng hiếm trọng số càng cao
counts = np.bincount(y_tr, minlength=len(lab2id))
weights = counts.sum() / (len(counts) * np.maximum(counts, 1))
criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float).to(device))
print("Trọng số lớp:", dict(zip(lab2id, weights.round(2))))

In [ ]:
import copy

def evaluate(model, dl):
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss_sum += criterion(out, yb).item() * len(yb)
            correct += (out.argmax(1) == yb).sum().item()
            total += len(yb)
    return loss_sum / total, correct / total

# khởi tạo lại model để chạy lại cell này không bị train tiếp từ lần trước
model_txt = TextCNN(len(itos), len(lab2id), pad_idx=stoi[PAD]).to(device)
optimizer = torch.optim.Adam(model_txt.parameters(), lr=1e-3, weight_decay=1e-4)

EPOCHS, PATIENCE = 100, 20
best_acc, best_loss, best_state, wait = 0.0, 1e9, None, 0
history = []

for ep in range(1, EPOCHS + 1):
    model_txt.train()
    tr_loss, n = 0.0, 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model_txt(xb), yb)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item() * len(yb); n += len(yb)

    va_loss, va_acc = evaluate(model_txt, val_dl)
    history.append((ep, tr_loss / n, va_loss, va_acc))
    if ep % 10 == 0 or ep == 1:
        print(f"ep {ep:3d} | train loss {tr_loss/n:.3f} | val loss {va_loss:.3f} | val acc {va_acc:.3f}")

    # chọn model tốt nhất: acc cao hơn, hoặc acc bằng nhưng loss thấp hơn
    if va_acc > best_acc or (va_acc == best_acc and va_loss < best_loss):
        best_acc, best_loss, wait = va_acc, va_loss, 0
        best_state = copy.deepcopy(model_txt.state_dict())
    else:
        wait += 1
        if wait >= PATIENCE:
            print("Dừng sớm ở epoch", ep); break

model_txt.load_state_dict(best_state)
print(f"Tốt nhất: val acc {best_acc:.3f}, val loss {best_loss:.3f}")

In [ ]:
from sklearn.metrics import classification_report

model_txt.eval()
with torch.no_grad():
    pred = model_txt(Xva_t.to(device)).argmax(1).cpu().numpy()

names = [k for k, _ in sorted(lab2id.items(), key=lambda kv: kv[1])]
print(classification_report(y_va, pred, labels=list(range(len(names))),
                            target_names=names, zero_division=0))

In [ ]:
import json
torch.save(best_state, "/kaggle/working/textcnn.pt")
json.dump({"itos": itos, "lab2id": lab2id, "max_len": max_len},
          open("/kaggle/working/textcnn_meta.json", "w"), ensure_ascii=False)

In [ ]:
import os
for f in ["textcnn.pt", "textcnn_meta.json"]:
    p = f"/kaggle/working/{f}"
    print(f, "OK" if os.path.exists(p) else "CHƯA CÓ", os.path.getsize(p) if os.path.exists(p) else "")

In [ ]:
meta = json.load(open("/kaggle/working/textcnn_meta.json", encoding="utf-8"))
itos, lab2id, max_len = meta["itos"], meta["lab2id"], meta["max_len"]
stoi = {w: i for i, w in enumerate(itos)}

model_txt = TextCNN(len(itos), len(lab2id), pad_idx=stoi["<pad>"]).to(device)
model_txt.load_state_dict(torch.load("/kaggle/working/textcnn.pt", map_location=device))
model_txt.eval()

# 15. Đánh giá TextCNN 

In [ ]:
import numpy as np, torch
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

model_txt.eval()
with torch.no_grad():
    logits = model_txt(Xva_t.to(device))
    prob = torch.softmax(logits, dim=1).cpu().numpy()
pred = prob.argmax(1)

names = [k for k, _ in sorted(lab2id.items(), key=lambda kv: kv[1])]
print("Accuracy :", round(accuracy_score(y_va, pred), 3))
print("Macro-F1 :", round(f1_score(y_va, pred, average="macro", zero_division=0), 3))
print(classification_report(y_va, pred, labels=list(range(len(names))),
                            target_names=names, zero_division=0))

In [ ]:
import pandas as pd
cm = confusion_matrix(y_va, pred, labels=list(range(len(names))))
print(pd.DataFrame(cm, index=[f"thật:{n}" for n in names], columns=[f"đoán:{n}" for n in names]))

In [ ]:
pd.set_option("display.max_colwidth", 120)
wrong = va_df.copy()
wrong["đoán"] = [names[i] for i in pred]
wrong["conf"] = prob.max(1).round(2)
print(wrong[wrong["intent"] != wrong["đoán"]][["question", "intent", "đoán", "conf"]])

In [ ]:
import matplotlib.pyplot as plt
h = pd.DataFrame(history, columns=["ep", "train_loss", "val_loss", "val_acc"])
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
h.plot(x="ep", y=["train_loss", "val_loss"], ax=ax[0], title="Loss")
h.plot(x="ep", y="val_acc", ax=ax[1], title="Val accuracy")
plt.show()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# CHỈ để so sánh. Không dùng baseline này để tạo submission (thể lệ yêu cầu Intent = TextCNN).
clf_baseline = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), analyzer="char_wb"),
                             LogisticRegression(max_iter=1000, class_weight="balanced"))
clf_baseline.fit(tr_df["question"], tr_df["intent"])
print("TF-IDF+LR (baseline) acc:", round(accuracy_score(va_df["intent"], clf_baseline.predict(va_df["question"])), 3))

# 15b. Xuất TextCNN sang TFLite INT8 (ONNX)

Thể lệ: Intent model là TextCNN **lượng tử hóa INT8**, chạy bằng TFLite Micro, dung lượng ≤ 1,5 MB. Quy trình: PyTorch → ONNX → SavedModel (`onnx2tf`) → TFLite INT8. Nếu bước ONNX lỗi thì tự chuyển sang phương án dự phòng dựng lại mô hình bằng Keras. Cuối mục in danh sách toán tử để đối chiếu với bộ mã TFLite Micro của BTC.

In [ ]:
!pip install -q onnx onnxruntime onnx2tf sng4onnx onnxsim tf_keras psutil
!pip install -q onnx_graphsurgeon --extra-index-url https://pypi.ngc.nvidia.com

In [ ]:
import numpy as np, torch, onnxruntime as ort

ONNX_PATH = "/kaggle/working/textcnn.onnx"
model_txt.eval().cpu()
dummy = torch.zeros(1, max_len, dtype=torch.int32)      # batch cố định = 1, token id kiểu int32

kw = dict(input_names=["tokens"], output_names=["logits"], opset_version=13)
try:
    torch.onnx.export(model_txt, dummy, ONNX_PATH, dynamo=False, **kw)
except TypeError:
    torch.onnx.export(model_txt, dummy, ONNX_PATH, **kw)

# Kiểm tra ONNX == PyTorch trên tập val
sess = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])
with torch.no_grad():
    pt_pred = model_txt(torch.tensor(X_va, dtype=torch.long)).argmax(1).numpy()
ox_pred = np.array([sess.run(None, {"tokens": r.reshape(1, -1).astype(np.int32)})[0].argmax() for r in X_va])
print("ONNX khớp PyTorch:", (pt_pred == ox_pred).mean())      # phải ~1.0
model_txt.to(device)                                           # trả lại device cho các cell sau

In [ ]:
import os, subprocess, tensorflow as tf

def to_int8(make_converter, in_dtype):
    """Convert sang TFLite INT8. Thử INT8 thuần trước; nếu lỗi mới cho phép fallback float (và cảnh báo)."""
    def rep():
        for row in X_tr:                                       # chỉ dùng dữ liệu train
            yield [row.reshape(1, -1).astype(in_dtype)]
    for strict in (True, False):
        conv = make_converter()
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        conv.representative_dataset = rep
        if strict:
            conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
            conv.inference_output_type = tf.int8
        else:
            conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8, tf.lite.OpsSet.TFLITE_BUILTINS]
        try:
            m = conv.convert()
            print("Convert OK:", "INT8 thuần" if strict else "INT8 + fallback float (CHƯA đạt INT8 thuần)")
            return m
        except Exception as e:
            print(f"Convert lỗi (strict={strict}):", type(e).__name__, str(e)[:300])
    return None

# --- Phương án chính: ONNX -> SavedModel -> TFLite INT8 ---
tflite_model, route = None, None
SM_DIR = "/kaggle/working/textcnn_tf"
try:
    subprocess.run(["onnx2tf", "-i", ONNX_PATH, "-o", SM_DIR], check=True)
    sig = tf.saved_model.load(SM_DIR).signatures["serving_default"]
    in_dtype = list(sig.structured_input_signature[1].values())[0].dtype.as_numpy_dtype
    print("Input dtype của SavedModel:", in_dtype)
    tflite_model = to_int8(lambda: tf.lite.TFLiteConverter.from_saved_model(SM_DIR), in_dtype)
    route = "ONNX"
except Exception as e:
    print("Đường ONNX lỗi:", type(e).__name__, str(e)[:300])

In [ ]:
# --- Phương án dự phòng: dựng lại cùng kiến trúc bằng Keras, chép trọng số từ PyTorch ---
if tflite_model is None:
    from tensorflow.keras import layers, models
    EMB, NF = 64, 32                                           # khớp class TextCNN (mục 13)

    inp = layers.Input(shape=(max_len,), batch_size=1, dtype="int32", name="tokens")
    x = layers.Embedding(len(itos), EMB, name="emb")(inp)
    pools = []
    for i, k in enumerate(KERNEL_SIZES):
        c = layers.Conv1D(NF, k, activation="relu", name=f"conv{i}")(x)
        pools.append(layers.GlobalMaxPooling1D(name=f"pool{i}")(c))
    out_k = layers.Dense(len(lab2id), name="fc")(layers.Concatenate(name="cat")(pools))
    model_tf = models.Model(inp, out_k)

    sd = {k: v.detach().cpu().numpy() for k, v in model_txt.state_dict().items()}
    model_tf.get_layer("emb").set_weights([sd["embedding.weight"]])
    for i in range(len(KERNEL_SIZES)):
        model_tf.get_layer(f"conv{i}").set_weights([sd[f"convs.{i}.weight"].transpose(2, 1, 0), sd[f"convs.{i}.bias"]])
    model_tf.get_layer("fc").set_weights([sd["fc.weight"].T, sd["fc.bias"]])

    kr_pred = model_tf.predict(X_va.astype(np.int32), batch_size=1, verbose=0).argmax(1)
    print("Keras khớp PyTorch:", (kr_pred == pt_pred).mean())  # phải ~1.0
    tflite_model = to_int8(lambda: tf.lite.TFLiteConverter.from_keras_model(model_tf), np.int32)
    route = "Keras"
else:
    print("Đã có model từ đường ONNX, bỏ qua phương án dự phòng.")

In [ ]:
# --- Lưu, kiểm tra kích thước, độ chính xác và toán tử ---
TFLITE_TXT = "/kaggle/working/textcnn_int8.tflite"
if tflite_model is None:
    print("CHƯA tạo được TextCNN TFLite INT8 -> cần xử lý trước khi BTC kiểm tra (mục IV.6).")
else:
    open(TFLITE_TXT, "wb").write(tflite_model)
    size = os.path.getsize(TFLITE_TXT)
    print(f"[{route}] Kích thước: {size/1024:.1f} KB |", "OK (<= 1.5MB)" if size <= 1.5*1024*1024 else "VƯỢT 1.5MB")

    interp = tf.lite.Interpreter(model_content=tflite_model); interp.allocate_tensors()
    i_det, o_det = interp.get_input_details()[0], interp.get_output_details()[0]
    print("Input :", i_det["dtype"], i_det["shape"], "| Output:", o_det["dtype"], o_det["shape"])

    def tfl_predict(X):
        res = []
        for row in X:
            interp.set_tensor(i_det["index"], row.reshape(1, -1).astype(i_det["dtype"]))
            interp.invoke()
            res.append(interp.get_tensor(o_det["index"])[0].argmax())
        return np.array(res)

    q_pred = tfl_predict(X_va)
    print("Acc PyTorch :", round((pt_pred == y_va).mean(), 3))
    print("Acc TFLite8 :", round((q_pred == y_va).mean(), 3))
    print("Khớp PyTorch:", round((q_pred == pt_pred).mean(), 3))
    try:
        print("Toán tử:", sorted({d["op_name"] for d in interp._get_ops_details()}))
    except Exception as e:
        print("Không lấy được danh sách toán tử:", e)

# 16. Dự đoán intent_test

In [ ]:
import torch, numpy as np

# dùng lại hàm encode, tokenize, stoi, max_len từ bước vocabulary
model_txt.eval()
X_te = torch.tensor([encode(q, max_len) for q in test["question"]], dtype=torch.long).to(device)
with torch.no_grad():
    prob_te = torch.softmax(model_txt(X_te), dim=1).cpu().numpy()

id2lab = {i: l for l, i in lab2id.items()}
test["intent"] = [id2lab[i] for i in prob_te.argmax(1)]
test["conf"] = prob_te.max(1).round(2)

pd.set_option("display.max_colwidth", 100)
print(test[["question_id", "question", "intent", "conf"]])
print(test["intent"].value_counts())

In [ ]:
# Nhãn intent của submission PHẢI đến từ TextCNN (cell 91). Không ghi đè bằng mô hình khác.
assert set(test["intent"]) <= set(lab2id), "Có nhãn intent không thuộc tập nhãn hợp lệ"
print("Nguồn dự đoán intent: TextCNN | số câu:", len(test))
print(test["intent"].value_counts().to_dict())

In [ ]:
test[["question_id", "intent"]].to_csv("/kaggle/working/intent_pred.csv", index=False)

In [ ]:
print(sub["task_type"].value_counts())
print(sub.head(5))
print(sub.tail(5))

# 17. Xem định dạng submit

In [ ]:
print(sub["task_type"].value_counts())
pd.set_option("display.max_colwidth", 200)
for t in sub["task_type"].unique():
    print("\n==", t)
    print(sub[sub["task_type"] == t].head(5))

In [ ]:
# map question_id -> intent đã dự đoán (chỉ kiểm tra khớp ID; submission cuối được ghi ở mục 18)
intent_map = dict(zip(test["question_id"].astype(str), test["intent"]))

sub_out = sub.copy()
is_intent = sub_out["sample_id"].astype(str).isin(intent_map)
sub_out.loc[is_intent, "prediction"] = sub_out.loc[is_intent, "sample_id"].astype(str).map(intent_map)

print("Số dòng intent đã điền:", is_intent.sum(), "/", len(intent_map))
print(sub_out[is_intent].head())

# 18. Tạo submission 

In [ ]:
# File tạm chỉ chứa INTENT. KHÔNG nộp file này; submission.csv hoàn chỉnh được tạo ở cell sau.
sub_out.to_csv("/kaggle/working/submission_intent_only.csv", index=False)
print("Đã ghi file tạm submission_intent_only.csv (không dùng để nộp)")

In [ ]:
# Kiểm tra kết quả YOLO; không dùng biến pred vì biến này được TextCNN sử dụng ở các cell sau.
print(type(det_pred_df))
print(det_pred_df.shape)


In [ ]:
import json
import pandas as pd

# Ánh xạ dự đoán intent theo question_id.
intent_map = dict(zip(test["question_id"].astype(str), test["intent"]))

# Ánh xạ dự đoán detection theo image id (tối đa MAX_DET box / ảnh).
det_map = {}
for img_id, group in det_pred.groupby("id"):
    group = group.sort_values("conf", ascending=False).head(MAX_DET)
    items = []
    for _, row in group.iterrows():
        items.append({
            "class_id": int(row["cls"]),
            "score": round(min(max(float(row["conf"]), 0.0), 1.0), 4),
            "bbox": [round(float(row["x1"]), 1), round(float(row["y1"]), 1),
                     round(float(row["x2"]), 1), round(float(row["y2"]), 1)],
        })
    det_map[str(img_id)] = json.dumps(items, separators=(",", ":"))

def fill_prediction(row):
    sid = str(row["sample_id"])
    if row["task_type"] == "DETECTION":
        return det_map.get(sid, "[]")
    if row["task_type"] == "INTENT":
        return intent_map.get(sid, row["prediction"])
    return row["prediction"]

sub_out = sub.copy()
sub_out["prediction"] = sub_out.apply(fill_prediction, axis=1)
submission_path = "/kaggle/working/submission.csv"
sub_out.to_csv(submission_path, index=False)

chk = pd.read_csv(submission_path)
print("Submission:", chk.shape)
print("Đúng cột:", chk.columns.tolist() == sub.columns.tolist())
print("Prediction trống:", chk["prediction"].isna().sum())
print("Ảnh detection có box:",
      (chk.loc[chk.task_type == "DETECTION", "prediction"] != "[]").sum(),
      "/", (chk.task_type == "DETECTION").sum())
display(chk.head())

# 19. Check submission


In [ ]:
import json
import pandas as pd
from PIL import Image

chk = pd.read_csv("/kaggle/working/submission.csv", keep_default_na=False)
ref = pd.read_csv(root / "sample_submission.csv", keep_default_na=False)
problems = []

# 1) Cấu trúc file
print("Shape       :", chk.shape, "| mẫu:", ref.shape)
if chk.columns.tolist() != ref.columns.tolist():                     problems.append("cột khác mẫu")
if chk["sample_id"].tolist() != ref["sample_id"].tolist():           problems.append("sample_id không khớp/đúng thứ tự")
if chk["task_type"].tolist() != ref["task_type"].tolist():           problems.append("task_type không khớp")
if chk["sample_id"].duplicated().any():                              problems.append("trùng sample_id")
if (chk["prediction"].astype(str).str.strip() == "").any():          problems.append("prediction trống")

# 2) Phần INTENT
VALID_INTENTS = {"COUNT", "EXIST", "LIST", "LOCATION", "OUT_OF_SCOPE"}
it = chk[chk["task_type"] == "INTENT"]
bad_intent = sorted(set(it["prediction"]) - VALID_INTENTS)
print("\nINTENT:", len(it), "dòng | nhãn lạ:", bad_intent)
print(it["prediction"].value_counts().to_dict())
if bad_intent:                                                       problems.append(f"nhãn intent lạ: {bad_intent}")
if set(it["sample_id"]) != set(test["question_id"].astype(str)):     problems.append("ID INTENT không khớp intent_test.csv")

# 3) Phần DETECTION: parse JSON và kiểm tra từng box (theo kích thước THẬT của từng ảnh test)
img_size = {p.stem: Image.open(p).size for p in test_imgs}
det = chk[chk["task_type"] == "DETECTION"]
if set(det["sample_id"]) != set(img_size):                           problems.append("ID DETECTION không khớp ảnh test")

errs, n_box, bad_json, max_boxes = [], 0, 0, 0
for _, r in det.iterrows():
    try:
        items = json.loads(r["prediction"])
    except Exception:
        bad_json += 1
        continue
    max_boxes = max(max_boxes, len(items))
    W, H = img_size.get(str(r["sample_id"]), (None, None))
    for b in items:
        n_box += 1
        if set(b) != {"class_id", "score", "bbox"}:
            errs.append((r["sample_id"], "sai khoá", list(b))); continue
        if not (0 <= b["class_id"] < len(classes)):
            errs.append((r["sample_id"], "class_id ngoài khoảng", b["class_id"]))
        if not (0 <= b["score"] <= 1):
            errs.append((r["sample_id"], "score ngoài [0,1]", b["score"]))
        if len(b["bbox"]) != 4:
            errs.append((r["sample_id"], "bbox không đủ 4 số", b["bbox"])); continue
        x1, y1, x2, y2 = b["bbox"]
        if x2 <= x1 or y2 <= y1:
            errs.append((r["sample_id"], "bbox sai (x2<=x1 hoặc y2<=y1)", b["bbox"]))
        if W and (x1 < 0 or y1 < 0 or x2 > W or y2 > H):
            errs.append((r["sample_id"], f"bbox vượt ảnh {W}x{H}", b["bbox"]))

print("\nDETECTION:", len(det), "dòng | tổng box:", n_box, "| JSON lỗi:", bad_json,
      "| số box tối đa / ảnh:", max_boxes)
print("Số lỗi box:", len(errs))
for e in errs[:10]:
    print("  ", e)
if bad_json:        problems.append("có prediction DETECTION không phải JSON")
if errs:            problems.append("có box sai định dạng")
if max_boxes > 100: problems.append("có ảnh > 100 bbox (sẽ bị từ chối chấm)")

print("\n=> KẾT LUẬN:", "SẴN SÀNG NỘP" if not problems else "CHƯA ĐẠT: " + "; ".join(problems))

# 20. Kiểm tra ràng buộc triển khai (mục V thể lệ)

In [ ]:
import os
MB = 1024 * 1024
g = globals()
print("=== Detection ===")
print("imgsz            :", g.get("IMGSZ"), "(<= 320x240; khuyến nghị 96/128/160)")
print("max_det / ảnh    :", g.get("MAX_DET"), "(<= 100)")
try:
    print("số tham số       :", f"{sum(p.numel() for p in best_model.model.parameters()):,}", "(<= 200,000)")
except Exception:
    print("số tham số       : chưa có best_model")
p = g.get("DET_TFLITE")
print("TFLite INT8      :", f"{os.path.getsize(p)/MB:.3f} MB (<= 1.5 MB)" if p and os.path.exists(p) else "CHƯA tạo được")

print("\n=== Intent TextCNN ===")
print("số tham số       :", f"{sum(p.numel() for p in model_txt.parameters()):,}")
p = "/kaggle/working/textcnn_int8.tflite"
print("TFLite INT8      :", f"{os.path.getsize(p)/MB:.3f} MB (<= 1.5 MB)" if os.path.exists(p) else "CHƯA tạo được")
print("max_len / vocab  :", max_len, "/", len(itos), "(cần để cài tokenizer trên ESP32)")

print("\n=== Khai báo (mục VII) ===")
print("Dữ liệu ngoài    : KHÔNG")
print("Pretrained       : KHÔNG (YOLO khởi tạo từ YAML, TextCNN train từ đầu)")
print("Dùng tập test để train / pseudo-label: KHÔNG")